In [5]:
import sys
import torch
import torchvision

from PIL import Image
from torchvision import transforms

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    
print( 'device: ', device )

#torchvision으로 resnet50 다운 방법
model = torchvision.models.resnet50( weights=torchvision.models.ResNet50_Weights.DEFAULT )
model.eval()

filename = "./data/resnet/cat.jpg"
input_image = Image.open(filename)

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

input_batch = input_batch.to(device)
model.to(device)

with torch.no_grad():
    output = model(input_batch)
# Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes
# print(output[0])
# The output has unnormalized scores. To get probabilities, you can run a softmax on it.
probabilities = torch.nn.functional.softmax(output[0], dim=0)
# print(probabilities)

# Read the categories
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]
# Show top categories per image
top5_prob, top5_catid = torch.topk(probabilities, 5)
print( top5_prob )
print( top5_catid )
for i in range(top5_prob.size(0)):
    print(categories[top5_catid[i]], top5_prob[i].item())


device:  cuda


FileNotFoundError: [Errno 2] No such file or directory: 'imagenet_classes.txt'